# 📘 Pandas for Data Engineering
## Databricks Notebook — Single-Node Data Wrangling

**What you'll learn:**
- When to use Pandas vs PySpark (decision framework)
- Data cleaning: NULLs, strings, types, duplicates
- Transformations: apply, groupby, merge, pivot, pipe
- Data validation & profiling
- File I/O, time series, memory management
- Pandas + Spark integration (toPandas, createDataFrame, Pandas UDFs)

**Prerequisites:** Notebooks 01-08 (techmart_dw)

**⚠️ Key rule:** Pandas is single-node. Always `.limit()` before `.toPandas()` on large tables.

---

---
# 📗 Section 1: When to Use Pandas vs PySpark

| Criteria | Pandas | PySpark |
|---|---|---|
| Data size | < 1 GB | Any size |
| Processing | Single machine | Distributed cluster |
| Use case | Prototyping, validation, small transforms | Production pipelines |
| Row-level logic | ✅ Easy (apply, iterrows) | ⚠️ Harder (UDFs) |
| Speed on small data | ⚡ Faster (no overhead) | 🐌 Slower (JVM startup) |
| Memory | Limited by driver RAM | Limited by cluster |

**Decision framework:**
1. Data fits in memory AND complex row logic → **Pandas**
2. Data > 1GB OR production pipeline → **PySpark**
3. Prototyping/exploring → **Pandas** (then convert to PySpark)
4. Validation/profiling of samples → **Pandas**

In [0]:
import pandas as pd
import numpy as np

# Enable Arrow for faster toPandas()
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

# Load sample data from Spark into Pandas
customers_pdf = spark.table("techmart_dw.customers").limit(5000).toPandas()
orders_pdf = spark.table("techmart_dw.orders").limit(10000).toPandas()
products_pdf = spark.table("techmart_dw.products").toPandas()

print(f"customers: {customers_pdf.shape}")
print(f"orders: {orders_pdf.shape}")
print(f"products: {products_pdf.shape}")

---
# 📗 Section 2: Pandas Fundamentals for DE

In [0]:
# Basic exploration
print("=== customers_pdf ===")
print(f"Shape: {customers_pdf.shape}")
print(f"\nDtypes:")
print(customers_pdf.dtypes)
print(f"\nFirst 3 rows:")
customers_pdf.head(3)

In [0]:
# info() — quick overview of nulls and types
customers_pdf.info()

In [0]:
# describe() — statistics for numeric columns
customers_pdf.describe()

In [0]:
# Selecting and filtering
# Select columns
subset = customers_pdf[["customer_id", "first_name", "email", "customer_segment"]]

# Filter rows
premium = customers_pdf[customers_pdf["customer_segment"] == "Premium"]
print(f"Premium customers: {len(premium)}")

# Multiple conditions (use & for AND, | for OR, ~ for NOT)
high_value_active = customers_pdf[
    (customers_pdf["lifetime_value"] > 30000) & 
    (customers_pdf["is_active"] == True)
]
print(f"High-value active: {len(high_value_active)}")

---
# 📗 Section 3: Data Cleaning with Pandas

## NULL Handling

In [0]:
# NULL analysis
print("NULL counts per column:")
null_counts = customers_pdf.isnull().sum()
null_pcts = (null_counts / len(customers_pdf) * 100).round(2)
null_report = pd.DataFrame({"nulls": null_counts, "pct": null_pcts})
print(null_report[null_report["nulls"] > 0])

In [0]:
# fillna strategies
df = customers_pdf.copy()

# Fill with constant
df["email"] = df["email"].fillna("unknown@placeholder.com")

# Fill with mode (most common value)
df["phone"] = df["phone"].fillna(df["phone"].mode()[0])

# Verify
print(f"Email nulls after fill: {df['email'].isnull().sum()}")
print(f"Phone nulls after fill: {df['phone'].isnull().sum()}")

## String Cleaning

In [0]:
# String operations (vectorized with .str accessor)
df = customers_pdf.copy()

# Check the mess: inconsistent casing
print("First name samples (notice inconsistency):")
print(df["first_name"].head(20).tolist())

# Clean: strip whitespace, title case
df["first_name_clean"] = df["first_name"].str.strip().str.title()

# Email domain extraction
df["email_domain"] = df["email"].str.split("@").str[1]

# Pattern matching
gmail_users = df[df["email"].str.contains("gmail", na=False)]
print(f"\nGmail users: {len(gmail_users)}")

# Validation with regex
valid_email = df["email"].str.match(r'^[^@]+@[^@]+\.[^@]+', na=False)
print(f"Valid email format: {valid_email.sum()}/{len(df)}")

## Type Conversion & Duplicates

In [0]:
# Safe type conversion with errors='coerce'
df = orders_pdf.copy()

# Convert total_amount to float (handles any bad values gracefully)
df["total_amount"] = pd.to_numeric(df["total_amount"], errors="coerce")
print(f"NaN after coerce: {df['total_amount'].isna().sum()}")

# Datetime conversion
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
print(f"Date dtype: {df['order_date'].dtype}")

In [0]:
# Duplicate detection
df = customers_pdf.copy()
print(f"Total rows: {len(df)}")

# Exact duplicates
exact_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes}")

# Duplicates on specific columns (e.g., same email)
email_dupes = df[df.duplicated(subset=["email"], keep=False)]
print(f"Duplicate emails: {len(email_dupes)}")

# Keep last occurrence
deduped = df.drop_duplicates(subset=["email"], keep="last")
print(f"After dedup: {len(deduped)}")

In [0]:
# ============================================
# 🎯 YOUR TURN: Clean the Customers DataFrame
# ============================================
# 1. Load customers (first 5000) into pandas
# 2. Fill NULL emails with 'no_email@unknown.com'
# 3. Strip and title-case first_name
# 4. Remove exact duplicates
# 5. Remove email duplicates (keep first)
# 6. Print before/after row counts
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
df = spark.table("techmart_dw.customers").limit(5000).toPandas()
print(f"Before: {len(df)} rows")

df["email"] = df["email"].fillna("no_email@unknown.com")
df["first_name"] = df["first_name"].str.strip().str.title()
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["email"], keep="first")
print(f"After: {len(df)} rows")
print(f"NULL emails: {df['email'].isnull().sum()}")

---
# 📗 Section 4: Data Transformation

## apply(), groupby(), merge()

In [0]:
# apply() for row-level logic
df = orders_pdf.copy()

# Classify orders using apply with a function
def classify_order(row):
    if row["total_amount"] > 4000:
        return "Premium"
    elif row["total_amount"] > 2000:
        return "Standard"
    else:
        return "Basic"

df["order_tier"] = df.apply(classify_order, axis=1)
print(df["order_tier"].value_counts())

In [0]:
# groupby with named aggregation
order_summary = orders_pdf.groupby("status").agg(
    order_count=("order_id", "count"),
    total_revenue=("total_amount", "sum"),
    avg_order=("total_amount", "mean"),
    max_order=("total_amount", "max")
).round(2)

print(order_summary.sort_values("total_revenue", ascending=False))

In [0]:
# merge (join) — like SQL JOIN
# Join orders with customers
merged = orders_pdf.merge(
    customers_pdf[["customer_id", "first_name", "customer_segment"]],
    on="customer_id",
    how="left"
)
print(f"Merged shape: {merged.shape}")
print(merged[["order_id", "customer_id", "first_name", "customer_segment", "total_amount"]].head())

In [0]:
# pivot_table — reshape data
pivot = orders_pdf.pivot_table(
    values="total_amount",
    index="status",
    columns="payment_method",
    aggfunc="count",
    fill_value=0
)
print(pivot)

In [0]:
# pipe() for composable transformations
def add_date_parts(df):
    df = df.copy()
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["year"] = df["order_date"].dt.year
    df["month"] = df["order_date"].dt.month
    return df

def add_tier(df):
    df = df.copy()
    df["tier"] = pd.cut(df["total_amount"], bins=[0, 1000, 3000, float("inf")], labels=["Low", "Mid", "High"])
    return df

# Chain transformations cleanly
result = (orders_pdf
    .pipe(add_date_parts)
    .pipe(add_tier)
)
print(result[["order_id", "year", "month", "total_amount", "tier"]].head())

In [0]:
# ============================================
# 🎯 YOUR TURN: Customer Summary Report
# ============================================
# Using orders_pdf and customers_pdf:
# 1. Merge orders with customers on customer_id
# 2. Group by customer_segment
# 3. Calculate: order_count, total_revenue, avg_order_value, unique_customers
# 4. Sort by total_revenue descending
# 5. Display the result
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
merged = orders_pdf.merge(customers_pdf[["customer_id", "customer_segment"]], on="customer_id", how="left")

summary = merged.groupby("customer_segment").agg(
    order_count=("order_id", "count"),
    total_revenue=("total_amount", "sum"),
    avg_order_value=("total_amount", "mean"),
    unique_customers=("customer_id", "nunique")
).round(2).sort_values("total_revenue", ascending=False)

print(summary)

---
# 📗 Section 5: Data Validation & Profiling

In [0]:
# Reusable data profiling function
def profile_dataframe(df, name="DataFrame"):
    """Generate a comprehensive data profile."""
    print(f"{'='*60}")
    print(f"DATA PROFILE: {name}")
    print(f"{'='*60}")
    print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
    print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"Duplicates: {df.duplicated().sum()}")
    
    print(f"\n{'Column':<20} {'Type':<12} {'Nulls':<8} {'Null%':<8} {'Unique':<8} {'Top Value'}")
    print("-" * 75)
    
    for col in df.columns:
        dtype = str(df[col].dtype)[:10]
        nulls = df[col].isnull().sum()
        null_pct = f"{nulls/len(df)*100:.1f}%"
        unique = df[col].nunique()
        top = str(df[col].mode().iloc[0])[:20] if len(df[col].dropna()) > 0 else "N/A"
        print(f"{col:<20} {dtype:<12} {nulls:<8} {null_pct:<8} {unique:<8} {top}")
    
    print(f"{'='*60}")

profile_dataframe(customers_pdf, "techmart_dw.customers")

In [0]:
# Referential integrity check
def check_referential_integrity(child_df, parent_df, child_col, parent_col):
    child_keys = set(child_df[child_col].dropna().unique())
    parent_keys = set(parent_df[parent_col].dropna().unique())
    
    orphans = child_keys - parent_keys
    unused = parent_keys - child_keys
    
    print(f"Referential Integrity: {child_col} → {parent_col}")
    print(f"  Child keys: {len(child_keys):,}")
    print(f"  Parent keys: {len(parent_keys):,}")
    print(f"  Orphans (in child, not parent): {len(orphans)}")
    print(f"  Unused (in parent, not child): {len(unused)}")
    return len(orphans) == 0

# Check: do all order customer_ids exist in customers?
check_referential_integrity(orders_pdf, customers_pdf, "customer_id", "customer_id")

In [0]:
# ============================================
# 🎯 YOUR TURN: Profile & Validate Orders
# ============================================
# 1. Run profile_dataframe on orders_pdf
# 2. Check referential integrity: orders.customer_id → customers.customer_id
# 3. Validate: total_amount > 0 (what % fail?)
# 4. Validate: order_date is not in the future
# 5. Find statistical outliers in total_amount (> 3 std devs from mean)
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
profile_dataframe(orders_pdf, "techmart_dw.orders")

print("\n--- Referential Integrity ---")
check_referential_integrity(orders_pdf, customers_pdf, "customer_id", "customer_id")

print("\n--- Value Validations ---")
negative_amount = (orders_pdf["total_amount"] <= 0).sum()
print(f"total_amount <= 0: {negative_amount} ({negative_amount/len(orders_pdf)*100:.2f}%)")

orders_pdf["order_date"] = pd.to_datetime(orders_pdf["order_date"])
future_dates = (orders_pdf["order_date"] > pd.Timestamp.now()).sum()
print(f"Future dates: {future_dates}")

print("\n--- Outlier Detection ---")
mean = orders_pdf["total_amount"].mean()
std = orders_pdf["total_amount"].std()
outliers = orders_pdf[abs(orders_pdf["total_amount"] - mean) > 3 * std]
print(f"Outliers (>3σ): {len(outliers)} ({len(outliers)/len(orders_pdf)*100:.2f}%)")

---
# 📗 Section 6: File I/O for DE

In [0]:
# CSV read/write with options
# Simulate reading a messy CSV
import io

messy_csv = """id,name,amount,date
1,"John Doe",150.50,2024-01-15
2,"Jane Smith",,2024-01-16
3,Bob Wilson,bad_number,2024-01-17
4,"Alice Brown",200.00,
5,"Charlie",350.75,2024-01-19"""

df = pd.read_csv(io.StringIO(messy_csv))
print("Raw CSV:")
print(df)
print(f"\nDtypes: {df.dtypes.to_dict()}")

# Clean it
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df["date"] = pd.to_datetime(df["date"], errors="coerce")
print(f"\nCleaned:")
print(df)

In [0]:
# JSON handling
import json

# Nested JSON (common in API responses)
json_data = [
    {"id": 1, "name": "Order A", "items": [{"sku": "X1", "qty": 2}, {"sku": "X2", "qty": 1}]},
    {"id": 2, "name": "Order B", "items": [{"sku": "X3", "qty": 5}]},
]

# Flatten nested JSON
from pandas import json_normalize
flat = json_normalize(json_data, record_path="items", meta=["id", "name"])
print("Flattened JSON:")
print(flat)

---
# 📗 Section 7: Time Series Operations

In [0]:
# Time series with orders
df = orders_pdf.copy()
df["order_date"] = pd.to_datetime(df["order_date"])
df = df.set_index("order_date").sort_index()

# Resample: daily → weekly → monthly
daily = df["total_amount"].resample("D").agg(["sum", "count"])
weekly = df["total_amount"].resample("W").agg(["sum", "count"])
monthly = df["total_amount"].resample("M").agg(["sum", "count"])

print("Monthly summary:")
print(monthly.tail(6))

In [0]:
# Rolling (moving average) and shift (lag)
daily_revenue = df["total_amount"].resample("D").sum().reset_index()
daily_revenue.columns = ["date", "revenue"]

# 7-day moving average
daily_revenue["ma_7d"] = daily_revenue["revenue"].rolling(7).mean()

# Lag: previous day
daily_revenue["prev_day"] = daily_revenue["revenue"].shift(1)

# Growth
daily_revenue["daily_growth"] = (daily_revenue["revenue"] - daily_revenue["prev_day"]) / daily_revenue["prev_day"] * 100

print(daily_revenue.dropna().head(10))

---
# 📗 Section 8: Pandas + Spark Integration

In [0]:
# Spark → Pandas → Spark roundtrip
# Step 1: Read from Spark
spark_df = spark.table("techmart_dw.products")
pdf = spark_df.toPandas()
print(f"Pandas DataFrame: {pdf.shape}")

# Step 2: Do complex pandas operations
pdf["profit_margin"] = ((pdf["price"] - pdf["cost"]) / pdf["price"] * 100).round(2)
pdf["price_band"] = pd.cut(pdf["price"], bins=[0, 200, 800, 2000, float("inf")], 
                           labels=["Budget", "Mid", "Premium", "Luxury"])

# Step 3: Write back to Spark
enriched_spark_df = spark.createDataFrame(pdf)
enriched_spark_df.write.format("delta").mode("overwrite").saveAsTable("techmart_dw.products_enriched")
print(f"Written back to Spark: {spark.table('techmart_dw.products_enriched').count()} rows")

In [0]:
# pandas_on_spark (pyspark.pandas) — pandas API on distributed Spark
import pyspark.pandas as ps

# This looks like pandas but runs on Spark (distributed!)
psdf = ps.read_table("techmart_dw.orders")
print(f"Type: {type(psdf)}")
print(f"Shape: {psdf.shape}")

# Pandas-like operations, but distributed
result = psdf.groupby("status")["total_amount"].agg(["count", "sum", "mean"])
print(result)

---
# 📗 Section 9: Memory Management

In [0]:
# Memory usage analysis
df = customers_pdf.copy()
print("Memory usage BEFORE optimization:")
mem_before = df.memory_usage(deep=True)
print(f"  Total: {mem_before.sum() / 1024**2:.2f} MB")
print(mem_before)

In [0]:
# Optimize: category dtype for low-cardinality strings
df_opt = df.copy()

# Convert low-cardinality columns to category
for col in ["customer_segment", "city", "state", "country"]:
    if col in df_opt.columns:
        df_opt[col] = df_opt[col].astype("category")

# Downcast numerics
for col in df_opt.select_dtypes(include=["int64"]).columns:
    df_opt[col] = pd.to_numeric(df_opt[col], downcast="integer")

for col in df_opt.select_dtypes(include=["float64"]).columns:
    df_opt[col] = pd.to_numeric(df_opt[col], downcast="float")

mem_after = df_opt.memory_usage(deep=True)
print("Memory usage AFTER optimization:")
print(f"  Total: {mem_after.sum() / 1024**2:.2f} MB")
print(f"  Savings: {(1 - mem_after.sum()/mem_before.sum())*100:.1f}%")

In [0]:
# Chunked reading pattern (for large files)
# Simulating with our data
import io

# Create a "large" CSV in memory
large_csv = customers_pdf.to_csv(index=False)

# Process in chunks
chunk_size = 1000
total_rows = 0
segment_counts = {}

for chunk in pd.read_csv(io.StringIO(large_csv), chunksize=chunk_size):
    total_rows += len(chunk)
    for seg, count in chunk["customer_segment"].value_counts().items():
        segment_counts[seg] = segment_counts.get(seg, 0) + count

print(f"Processed {total_rows:,} rows in chunks of {chunk_size}")
print(f"Segment distribution: {segment_counts}")

---
# 🚀 Section 10: Mini Projects

## Project 1: Data Quality Report Generator

In [0]:
def generate_quality_report(df, name="Dataset"):
    """Generate a comprehensive data quality report."""
    report = []
    total_rows = len(df)
    
    for col in df.columns:
        null_count = df[col].isnull().sum()
        null_pct = null_count / total_rows * 100
        unique_count = df[col].nunique()
        unique_pct = unique_count / total_rows * 100
        
        # Determine pass/fail
        status = "PASS"
        if null_pct > 10:
            status = "WARN"
        if null_pct > 50:
            status = "FAIL"
        
        report.append({
            "column": col,
            "dtype": str(df[col].dtype),
            "null_count": null_count,
            "null_pct": round(null_pct, 2),
            "unique_count": unique_count,
            "unique_pct": round(unique_pct, 2),
            "status": status
        })
    
    report_df = pd.DataFrame(report)
    
    # Summary
    passed = (report_df["status"] == "PASS").sum()
    warned = (report_df["status"] == "WARN").sum()
    failed = (report_df["status"] == "FAIL").sum()
    
    print(f"{'='*60}")
    print(f"QUALITY REPORT: {name}")
    print(f"{'='*60}")
    print(f"Rows: {total_rows:,} | Columns: {len(df.columns)}")
    print(f"Status: ✅ {passed} PASS | ⚠️ {warned} WARN | ❌ {failed} FAIL")
    print(f"{'='*60}")
    print(report_df.to_string(index=False))
    
    return report_df

# Run it
quality = generate_quality_report(customers_pdf, "customers")

## Project 2: Schema Comparison Tool

In [0]:
def compare_schemas(df1, df2, name1="Source", name2="Target"):
    """Compare schemas of two DataFrames."""
    cols1 = set(df1.columns)
    cols2 = set(df2.columns)
    
    comparison = []
    all_cols = sorted(cols1 | cols2)
    
    for col in all_cols:
        in_1 = col in cols1
        in_2 = col in cols2
        
        if in_1 and in_2:
            type1 = str(df1[col].dtype)
            type2 = str(df2[col].dtype)
            status = "match" if type1 == type2 else "type_mismatch"
        elif in_1:
            type1, type2 = str(df1[col].dtype), "N/A"
            status = "only_in_source"
        else:
            type1, type2 = "N/A", str(df2[col].dtype)
            status = "only_in_target"
        
        comparison.append({"column": col, f"{name1}_type": type1, f"{name2}_type": type2, "status": status})
    
    result = pd.DataFrame(comparison)
    print(f"Schema Comparison: {name1} vs {name2}")
    print(f"  Matches: {(result['status']=='match').sum()}")
    print(f"  Mismatches: {(result['status']=='type_mismatch').sum()}")
    print(f"  Only in {name1}: {(result['status']=='only_in_source').sum()}")
    print(f"  Only in {name2}: {(result['status']=='only_in_target').sum()}")
    print(result[result["status"] != "match"].to_string(index=False))
    return result

# Compare customers with a modified version
modified = customers_pdf.copy()
modified["new_col"] = "test"
modified = modified.drop(columns=["metadata"])
modified["lifetime_value"] = modified["lifetime_value"].astype(str)

compare_schemas(customers_pdf, modified, "Original", "Modified")

## Project 3: Reconciliation Report

In [0]:
def reconcile(source_df, target_df, key_col, compare_cols):
    """Compare source vs target and generate reconciliation report."""
    merged = source_df.merge(target_df, on=key_col, how="outer", suffixes=("_src", "_tgt"), indicator=True)
    
    only_source = merged[merged["_merge"] == "left_only"]
    only_target = merged[merged["_merge"] == "right_only"]
    both = merged[merged["_merge"] == "both"]
    
    # Check for value differences in matching records
    mismatches = []
    for col in compare_cols:
        src_col = f"{col}_src"
        tgt_col = f"{col}_tgt"
        if src_col in both.columns and tgt_col in both.columns:
            diff = both[both[src_col] != both[tgt_col]]
            if len(diff) > 0:
                mismatches.append({"column": col, "mismatch_count": len(diff)})
    
    print(f"{'='*50}")
    print(f"RECONCILIATION REPORT")
    print(f"{'='*50}")
    print(f"Source rows: {len(source_df):,}")
    print(f"Target rows: {len(target_df):,}")
    print(f"Only in source: {len(only_source):,}")
    print(f"Only in target: {len(only_target):,}")
    print(f"In both: {len(both):,}")
    if mismatches:
        print(f"\nValue mismatches:")
        for m in mismatches:
            print(f"  {m['column']}: {m['mismatch_count']} differences")
    print(f"{'='*50}")

# Demo: compare orders sample with a slightly different version
source = orders_pdf.head(1000)
target = orders_pdf.head(950)  # missing 50 rows
target.loc[target.index[:10], "total_amount"] = 0  # modify 10 rows

reconcile(source, target, "order_id", ["total_amount", "status"])

---
# 🏆 Section 11: Interview Questions

## Q1: When would you use pandas vs PySpark?

**Pandas:** Data < 1GB, complex row-level logic, prototyping, data validation, profiling samples.
**PySpark:** Data > 1GB, production pipelines, distributed processing, streaming.
**Hybrid:** Use pandas for validation/profiling on samples, PySpark for the actual pipeline.

## Q2: How do you handle memory issues with large pandas DataFrames?

1. **Chunked reading:** `pd.read_csv(chunksize=10000)` — process in batches
2. **Category dtype:** Convert low-cardinality strings (saves 90%+ memory)
3. **Downcast numerics:** `pd.to_numeric(s, downcast='integer')`
4. **Select only needed columns:** Don't load what you don't need
5. **Process and discard:** `del df; gc.collect()`
6. **Use PySpark instead:** If data doesn't fit, switch to distributed

## Q3: Write a data validation function

See Project 1 above — `generate_quality_report()` checks nulls, types, uniqueness, and returns pass/fail per column.

## Q4: How do you profile a new dataset?

1. `df.shape` — row/column counts
2. `df.dtypes` — column types
3. `df.isnull().sum()` — null analysis
4. `df.describe()` — numeric statistics
5. `df.nunique()` — cardinality
6. `df.duplicated().sum()` — duplicate check
7. Value distributions for key columns
8. Referential integrity checks against related tables

## Q5: Explain groupby-apply pattern

`df.groupby("key").apply(func)` applies a function to each group independently.
Use for complex per-group logic that can't be expressed with simple aggregations.
**Warning:** Slower than vectorized `.agg()` — use only when necessary.

## Q6: Mixed data types in a column?

Use `pd.to_numeric(series, errors='coerce')` — converts valid numbers, sets invalid to NaN.
Then investigate the NaN rows to understand the bad data.

## Q7: apply() vs map() vs applymap()?

- `apply()` — works on rows (axis=1) or columns (axis=0) of DataFrame
- `map()` — element-wise on a Series (or dict/function mapping)
- `applymap()` — element-wise on entire DataFrame (deprecated in favor of `map()` in pandas 2.0+)

## Q8: Optimize a 5GB CSV pipeline?

1. Read in chunks: `pd.read_csv(chunksize=100000)`
2. Select only needed columns: `usecols=['col1','col2']`
3. Specify dtypes upfront: `dtype={'id': 'int32', 'name': 'category'}`
4. Process each chunk, aggregate results
5. Or better: use PySpark/Dask for 5GB+ data

---
# 📗 Section 6: Advanced GroupBy & Window Operations

GroupBy is one of the most powerful Pandas tools for DE work — aggregating,
transforming, and filtering groups of data.

| Operation | Method | Use Case |
|-----------|--------|----------|
| Aggregate | `.agg()` | Sum, mean, count per group |
| Transform | `.transform()` | Broadcast group stats back to rows |
| Filter | `.filter()` | Keep/drop entire groups |
| Apply | `.apply()` | Custom function per group |

**Key insight:** `.transform()` returns same-shape DataFrame — perfect for
adding group-level stats as new columns without losing rows.

In [0]:
import pandas as pd
import numpy as np

# Sample sales data
np.random.seed(42)
df = pd.DataFrame({
    "region": np.random.choice(["North", "South", "East", "West"], 200),
    "product": np.random.choice(["Widget", "Gadget", "Doohickey"], 200),
    "sales": np.random.randint(100, 5000, 200),
    "units": np.random.randint(1, 50, 200),
    "month": np.random.choice(range(1, 13), 200),
})

# --- Multi-level aggregation ---
summary = df.groupby(["region", "product"]).agg(
    total_sales=("sales", "sum"),
    avg_sales=("sales", "mean"),
    total_units=("units", "sum"),
    order_count=("sales", "count"),
    max_sale=("sales", "max"),
).round(2).reset_index()
print("Multi-level aggregation:")
print(summary.head(8).to_string())

In [0]:
# --- transform(): add group stats as new columns ---
# Use case: "what % of region total does each order represent?"
df["region_total"] = df.groupby("region")["sales"].transform("sum")
df["pct_of_region"] = (df["sales"] / df["region_total"] * 100).round(2)

# Add rank within each region
df["rank_in_region"] = df.groupby("region")["sales"].rank(
    method="dense", ascending=False
).astype(int)

print("Transform example (first 10 rows):")
print(df[["region", "product", "sales", "region_total", "pct_of_region", "rank_in_region"]]
      .sort_values(["region", "rank_in_region"])
      .head(10)
      .to_string())

In [0]:
# --- filter(): keep only groups meeting a condition ---
# Keep only regions with total sales > 100,000
high_value_regions = df.groupby("region").filter(
    lambda g: g["sales"].sum() > 100_000
)
print(f"Regions with >100k sales: {high_value_regions['region'].unique()}")
print(f"Rows kept: {len(high_value_regions)} / {len(df)}")

# --- apply(): custom function per group ---
def top_n_products(group, n=2):
    """Return top N products by sales within a group."""
    return group.nlargest(n, "sales")

top_per_region = df.groupby("region").apply(top_n_products, n=2).reset_index(drop=True)
print(f"\nTop 2 orders per region ({len(top_per_region)} rows):")
print(top_per_region[["region", "product", "sales"]].to_string())

## 6.2 Window Functions with Pandas

Pandas rolling/expanding windows mirror SQL window functions — essential for
time-series and running totals in DE pipelines.

In [0]:
# Time-series window operations
dates = pd.date_range("2024-01-01", periods=30, freq="D")
ts = pd.DataFrame({
    "date": dates,
    "revenue": np.random.randint(1000, 10000, 30),
    "orders": np.random.randint(10, 100, 30),
})
ts = ts.set_index("date")

# Rolling windows
ts["revenue_7d_avg"] = ts["revenue"].rolling(7).mean().round(0)
ts["revenue_7d_sum"] = ts["revenue"].rolling(7).sum()
ts["revenue_7d_std"] = ts["revenue"].rolling(7).std().round(0)

# Expanding (cumulative)
ts["cumulative_revenue"] = ts["revenue"].expanding().sum()
ts["running_avg_orders"] = ts["orders"].expanding().mean().round(1)

# Lag / lead (shift)
ts["revenue_prev_day"] = ts["revenue"].shift(1)
ts["revenue_next_day"] = ts["revenue"].shift(-1)
ts["day_over_day_pct"] = ((ts["revenue"] / ts["revenue_prev_day"] - 1) * 100).round(1)

print("Window functions sample:")
print(ts[["revenue", "revenue_7d_avg", "cumulative_revenue", "day_over_day_pct"]].head(12).to_string())

---
# 📗 Section 7: Data Quality & Profiling Patterns

Production DE pipelines need automated data quality checks. Here are
reusable patterns for profiling, validating, and reporting on DataFrames.

In [0]:
def full_profile(df, name="DataFrame"):
    """
    Comprehensive data profiling — use before loading to Silver/Gold.
    Returns a summary DataFrame with quality metrics per column.
    """
    rows = len(df)
    profile = []
    for col in df.columns:
        s = df[col]
        null_count = s.isna().sum()
        unique_count = s.nunique()
        dtype = str(s.dtype)
        
        row = {
            "column": col,
            "dtype": dtype,
            "null_count": null_count,
            "null_pct": round(null_count / rows * 100, 1),
            "unique_count": unique_count,
            "unique_pct": round(unique_count / rows * 100, 1),
            "is_constant": unique_count == 1,
            "is_id_candidate": unique_count == rows and null_count == 0,
        }
        # Numeric stats
        if pd.api.types.is_numeric_dtype(s):
            row.update({
                "min": s.min(), "max": s.max(),
                "mean": round(s.mean(), 2),
                "median": s.median(),
                "std": round(s.std(), 2),
                "zeros": (s == 0).sum(),
                "negatives": (s < 0).sum(),
            })
        # String stats
        elif pd.api.types.is_string_dtype(s) or s.dtype == object:
            row.update({
                "min_len": s.dropna().str.len().min(),
                "max_len": s.dropna().str.len().max(),
                "has_whitespace": s.dropna().str.contains(r"^\s|\s$", regex=True).any(),
                "top_value": s.value_counts().index[0] if len(s.dropna()) > 0 else None,
            })
        profile.append(row)
    
    result = pd.DataFrame(profile).set_index("column")
    print(f"\n📊 Profile: {name} ({rows:,} rows, {len(df.columns)} columns)")
    print(result.to_string())
    return result

# Test it
sample = pd.DataFrame({
    "id": range(1, 101),
    "name": [f"Customer {i}" for i in range(1, 101)],
    "amount": np.random.choice([None, 100.0, 200.0, -50.0, 0.0], 100),
    "status": np.random.choice(["active", "inactive", "active"], 100),
    "email": [f"user{i}@example.com" if i % 10 != 0 else None for i in range(100)],
})
profile_df = full_profile(sample, "customers")

In [0]:
# --- Expectation-style validation ---
class DataExpectations:
    """Lightweight Great Expectations-style validator for Pandas."""
    
    def __init__(self, df, name="DataFrame"):
        self.df = df
        self.name = name
        self.results = []
    
    def expect_no_nulls(self, col):
        nulls = self.df[col].isna().sum()
        passed = nulls == 0
        self.results.append({"check": f"no_nulls({col})", "passed": passed,
                              "detail": f"{nulls} nulls found"})
        return self
    
    def expect_unique(self, col):
        dupes = self.df[col].duplicated().sum()
        passed = dupes == 0
        self.results.append({"check": f"unique({col})", "passed": passed,
                              "detail": f"{dupes} duplicates found"})
        return self
    
    def expect_values_in(self, col, valid_values):
        invalid = ~self.df[col].isin(valid_values)
        count = invalid.sum()
        passed = count == 0
        self.results.append({"check": f"values_in({col})", "passed": passed,
                              "detail": f"{count} invalid values"})
        return self
    
    def expect_range(self, col, min_val, max_val):
        out_of_range = ((self.df[col] < min_val) | (self.df[col] > max_val)).sum()
        passed = out_of_range == 0
        self.results.append({"check": f"range({col}, {min_val}, {max_val})",
                              "passed": passed, "detail": f"{out_of_range} out of range"})
        return self
    
    def expect_row_count_between(self, min_rows, max_rows):
        n = len(self.df)
        passed = min_rows <= n <= max_rows
        self.results.append({"check": f"row_count_between({min_rows}, {max_rows})",
                              "passed": passed, "detail": f"actual: {n}"})
        return self
    
    def report(self):
        df = pd.DataFrame(self.results)
        passed = df["passed"].sum()
        total = len(df)
        print(f"\n✅ Validation Report: {self.name}")
        print(f"   {passed}/{total} checks passed")
        print(df.to_string(index=False))
        if passed < total:
            raise ValueError(f"Data quality failed: {total - passed} checks failed")
        return df

# Run validations
orders = pd.DataFrame({
    "order_id": range(1, 51),
    "amount": np.random.uniform(10, 1000, 50),
    "status": np.random.choice(["pending", "completed", "cancelled"], 50),
    "customer_id": np.random.randint(1, 20, 50),
})

(DataExpectations(orders, "orders")
 .expect_no_nulls("order_id")
 .expect_unique("order_id")
 .expect_values_in("status", ["pending", "completed", "cancelled"])
 .expect_range("amount", 0, 10000)
 .expect_row_count_between(1, 1000)
 .report())

---
# 📗 Section 8: Production Pandas Patterns

These patterns appear in real DE pipelines — memory optimization, chunked
processing, and safe type handling.

In [0]:
# --- Memory optimization ---
def optimize_dtypes(df):
    """
    Automatically downcast numeric types and convert low-cardinality
    strings to category. Can reduce memory by 50-80%.
    """
    original_mb = df.memory_usage(deep=True).sum() / 1024**2
    result = df.copy()
    
    for col in result.columns:
        col_type = result[col].dtype
        
        if col_type == "object":
            # Convert low-cardinality strings to category
            n_unique = result[col].nunique()
            n_total = len(result[col])
            if n_unique / n_total < 0.5:  # < 50% unique → category
                result[col] = result[col].astype("category")
        
        elif col_type in ["int64", "int32"]:
            col_min = result[col].min()
            col_max = result[col].max()
            if col_min >= 0:
                if col_max < 255:
                    result[col] = result[col].astype("uint8")
                elif col_max < 65535:
                    result[col] = result[col].astype("uint16")
                elif col_max < 4294967295:
                    result[col] = result[col].astype("uint32")
            else:
                if col_min > -128 and col_max < 127:
                    result[col] = result[col].astype("int8")
                elif col_min > -32768 and col_max < 32767:
                    result[col] = result[col].astype("int16")
                elif col_min > -2147483648 and col_max < 2147483647:
                    result[col] = result[col].astype("int32")
        
        elif col_type == "float64":
            result[col] = pd.to_numeric(result[col], downcast="float")
    
    optimized_mb = result.memory_usage(deep=True).sum() / 1024**2
    reduction = (1 - optimized_mb / original_mb) * 100
    print(f"Memory: {original_mb:.2f} MB → {optimized_mb:.2f} MB ({reduction:.1f}% reduction)")
    return result

# Test with a large-ish DataFrame
big_df = pd.DataFrame({
    "id": np.arange(100_000, dtype="int64"),
    "age": np.random.randint(18, 90, 100_000).astype("int64"),
    "score": np.random.uniform(0, 100, 100_000).astype("float64"),
    "category": np.random.choice(["A", "B", "C", "D"], 100_000),
    "flag": np.random.choice([0, 1], 100_000).astype("int64"),
})
optimized_df = optimize_dtypes(big_df)
print("\nDtype comparison:")
print(pd.DataFrame({"original": big_df.dtypes, "optimized": optimized_df.dtypes}))

In [0]:
# --- Chunked file processing (large CSV files) ---
import io

def process_large_csv_chunked(filepath_or_buffer, chunk_size=10_000,
                               transform_fn=None, filter_fn=None):
    """
    Process a large CSV in chunks to avoid OOM errors.
    Returns aggregated results.
    
    Args:
        filepath_or_buffer: path or file-like object
        chunk_size: rows per chunk
        transform_fn: optional function to apply to each chunk
        filter_fn: optional function to filter rows in each chunk
    """
    chunks_processed = 0
    total_rows = 0
    results = []
    
    reader = pd.read_csv(filepath_or_buffer, chunksize=chunk_size)
    
    for chunk in reader:
        chunks_processed += 1
        
        # Apply filter
        if filter_fn:
            chunk = chunk[filter_fn(chunk)]
        
        # Apply transform
        if transform_fn:
            chunk = transform_fn(chunk)
        
        total_rows += len(chunk)
        results.append(chunk)
        
        if chunks_processed % 10 == 0:
            print(f"  Processed {chunks_processed} chunks, {total_rows:,} rows...")
    
    print(f"✅ Done: {chunks_processed} chunks, {total_rows:,} total rows")
    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()

# Simulate with in-memory CSV
csv_data = "id,amount,status\n" + "\n".join(
    f"{i},{i*10},{['active','inactive'][i%2]}" for i in range(1, 1001)
)
result = process_large_csv_chunked(
    io.StringIO(csv_data),
    chunk_size=100,
    filter_fn=lambda df: df["status"] == "active",
    transform_fn=lambda df: df.assign(amount_doubled=df["amount"] * 2),
)
print(f"\nResult shape: {result.shape}")
print(result.head(3))

## Common Mistakes & Edge Cases

These are the most frequent Pandas bugs in DE pipelines:

In [0]:
# ❌ MISTAKE 1: SettingWithCopyWarning — modifying a slice
df = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})
subset = df[df["a"] > 1]  # This is a VIEW, not a copy!

# ❌ Wrong — may not modify original, raises warning
# subset["b"] = 99

# ✅ Correct — use .copy() or .loc
subset = df[df["a"] > 1].copy()
subset["b"] = 99
print("After copy+modify:", subset)

# ❌ MISTAKE 2: Chained indexing
# df["col1"]["row"] = value  # Unpredictable!
# ✅ Correct
df.loc[df["a"] == 2, "b"] = 99
print("After .loc assign:", df)

# ❌ MISTAKE 3: iterrows() on large DataFrames (very slow)
# for idx, row in df.iterrows():  # O(n) Python loop
#     df.at[idx, "c"] = row["a"] + row["b"]

# ✅ Correct — vectorized
df["c"] = df["a"] + df["b"]
print("Vectorized result:", df)

# ❌ MISTAKE 4: Forgetting inplace=False (default)
df_sorted = df.sort_values("a")  # Returns new df
# df.sort_values("a")  # ← result discarded!
print("Sorted:", df_sorted)

In [0]:
# ❌ MISTAKE 5: merge() creating unexpected duplicates
left = pd.DataFrame({"id": [1, 2, 3], "val": ["a", "b", "c"]})
right = pd.DataFrame({"id": [1, 1, 2], "extra": ["x", "y", "z"]})  # id=1 appears twice!

merged = left.merge(right, on="id", how="left")
print(f"Left rows: {len(left)}, Right rows: {len(right)}, Merged rows: {len(merged)}")
print("⚠️  Merge created extra rows due to duplicates in right!")
print(merged)

# ✅ Fix: deduplicate before merge, or use validate parameter
try:
    safe_merge = left.merge(right.drop_duplicates("id"), on="id", how="left",
                            validate="1:1")
    print("\n✅ Safe merge:", safe_merge)
except Exception as e:
    print(f"\nValidation caught issue: {e}")

# ❌ MISTAKE 6: pd.to_datetime() silently returning NaT on bad dates
dates = ["2024-01-15", "not-a-date", "2024-13-01", "2024-02-30"]
parsed = pd.to_datetime(dates, errors="coerce")
print(f"\nParsed dates: {parsed.tolist()}")
print(f"Failed parses (NaT): {parsed.isna().sum()}")
# ✅ Always check for NaT after parsing!

---
# 📗 Section 9: Practice Exercises

Work through these exercises to solidify your Pandas DE skills.

In [0]:
# ============================================================
# EXERCISE 1: Data Cleaning Pipeline
# ============================================================
# Given this messy DataFrame, clean it:
# - Remove rows where order_id is null
# - Standardize status to lowercase, strip whitespace
# - Convert amount to float (some have "$" prefix)
# - Flag rows where amount < 0 as "invalid"
# - Deduplicate on order_id (keep first)

messy = pd.DataFrame({
    "order_id": [1, 2, None, 4, 4, 5],
    "status": ["  COMPLETED ", "pending", "SHIPPED", "cancelled", "cancelled", "COMPLETED"],
    "amount": ["$150.00", "200", "-50", "300.50", "300.50", "$75"],
})

# YOUR SOLUTION:
def clean_orders_pipeline(df):
    result = df.copy()
    # Step 1: Remove null order_ids
    result = result.dropna(subset=["order_id"])
    # Step 2: Standardize status
    result["status"] = result["status"].str.strip().str.lower()
    # Step 3: Parse amount
    result["amount"] = (result["amount"]
                        .str.replace("$", "", regex=False)
                        .astype(float))
    # Step 4: Flag invalid amounts
    result["is_valid"] = result["amount"] >= 0
    # Step 5: Deduplicate
    result = result.drop_duplicates(subset=["order_id"], keep="first")
    return result

cleaned = clean_orders_pipeline(messy)
print("Cleaned orders:")
print(cleaned.to_string(index=False))
assert len(cleaned) == 4, f"Expected 4 rows, got {len(cleaned)}"
assert cleaned["status"].str.contains(r"^\s|\s$").sum() == 0
print("\n✅ All assertions passed!")

In [0]:
# ============================================================
# EXERCISE 2: Revenue Attribution Report
# ============================================================
# Build a report showing:
# - Total revenue per region per month
# - Month-over-month growth %
# - Rank of each region within each month

np.random.seed(0)
sales = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=180, freq="D"),
    "region": np.random.choice(["North", "South", "East", "West"], 180),
    "revenue": np.random.randint(1000, 50000, 180),
})
sales["month"] = sales["date"].dt.to_period("M")

# Step 1: Aggregate by region + month
monthly = (sales.groupby(["region", "month"])["revenue"]
           .sum()
           .reset_index()
           .sort_values(["region", "month"]))

# Step 2: Month-over-month growth
monthly["prev_month_rev"] = monthly.groupby("region")["revenue"].shift(1)
monthly["mom_growth_pct"] = (
    (monthly["revenue"] / monthly["prev_month_rev"] - 1) * 100
).round(1)

# Step 3: Rank within each month
monthly["rank_in_month"] = (monthly.groupby("month")["revenue"]
                             .rank(method="dense", ascending=False)
                             .astype(int))

print("Revenue Attribution Report:")
print(monthly.to_string(index=False))

In [0]:
# ============================================================
# EXERCISE 3: Schema Drift Detection
# ============================================================
# Compare two DataFrames (e.g., yesterday vs today's data)
# and report: new columns, dropped columns, type changes

def detect_schema_drift(df_old, df_new, name=""):
    old_schema = dict(zip(df_old.columns, df_old.dtypes.astype(str)))
    new_schema = dict(zip(df_new.columns, df_new.dtypes.astype(str)))
    
    old_cols = set(old_schema.keys())
    new_cols = set(new_schema.keys())
    
    added = new_cols - old_cols
    dropped = old_cols - new_cols
    common = old_cols & new_cols
    type_changes = {
        col: (old_schema[col], new_schema[col])
        for col in common
        if old_schema[col] != new_schema[col]
    }
    
    print(f"\n🔍 Schema Drift Report: {name}")
    print(f"  Added columns:   {sorted(added) or 'None'}")
    print(f"  Dropped columns: {sorted(dropped) or 'None'}")
    print(f"  Type changes:    {type_changes or 'None'}")
    
    has_drift = bool(added or dropped or type_changes)
    print(f"  Drift detected:  {'⚠️  YES' if has_drift else '✅ NO'}")
    return has_drift

# Test
yesterday = pd.DataFrame({"id": [1,2], "name": ["a","b"], "amount": [1.0, 2.0]})
today = pd.DataFrame({"id": [1,2], "name": ["a","b"], "amount": [1, 2], "new_col": ["x","y"]})

detect_schema_drift(yesterday, today, "orders table")

---
# ✅ Notebook Complete!

**What was covered:**
- Pandas vs PySpark decision framework
- Data cleaning: NULLs, strings, types, duplicates
- Transformations: apply, groupby, merge, pivot, pipe
- Data validation & profiling (reusable functions)
- File I/O: CSV, JSON, Parquet
- Time series: resample, rolling, shift
- Pandas + Spark integration: toPandas, createDataFrame, pyspark.pandas
- Memory management: chunking, category dtype, downcasting
- 3 mini projects: Quality Report, Schema Comparison, Reconciliation
- 8 interview questions

**Next:** Notebook 10 — NumPy for Data Engineering

---